# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 20 kwietnia 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Losowanie przedmiotów w każdym przypadku powinno być tak ustawione, aby optymalne rozwiązanie wymagało wzięcia przynajmniej dwóch przedmiotów do każdego plecaka oraz zostawienia przynajmniej dwóch przedmiotów poza plecakami. W raporcie należy umieścić informację w jaki sposób zostało to zweryfikowane.
3. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków.
3. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania z podobnym interfejsem co tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia. Dobierz liczbę iteracji tak, aby czas działania był porównywalny do tabu search.


In [2]:
import Pkg
Pkg.add("Distributions")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed Rmath ─────────────────── v0.9.0
   Installed Rmath_jll ─────────────── v0.5.1+0
   Installed StatsFuns ─────────────── v1.5.2
   Installed PDMats ────────────────── v0.11.37
   Installed HypergeometricFunctions ─ v0.3.28
   Installed QuadGK ────────────────── v2.11.3
   Installed Distributions ─────────── v0.25.124
  Installing 1 artifacts
   Installed artifact Rmath                    121.9 KiB
    Updating `~/.julia/environments/v1.12/Project.toml`
  [31c24e10] + Distributions v0.25.124
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [31c24e10] + Distributions v0.25.124
  [34004b35] + HypergeometricFunctions v0.3.28
  [90014a1f] + PDMats v0.11.37
  [1fd47b50] + QuadGK v2.11.3
  [79098fc4] + Rmath v0.9.0
  [4c63d2b9] + StatsFuns v1.5.2
  [f50d1b31] + Rmath_jll v0.5.1+0
  [4607b0f0] + SuiteSparse
Precompiling packages...
    917.6 ms  ✓ Rmath_jll
   1258.3 ms  ✓ PDMats


In [3]:
using Distributions

function make_dzn(n::Int, capacity::Int)
    profits = rand(DiscreteUniform(10, 1000), n)
    weights = rand(DiscreteUniform(10, 100), n)
    content = """
    ITEM = _(1..$n);
    capacity = $capacity;
    profits = $profits;
    weights = $weights;
    """
    file = open("knapsack_generated.dzn", "w+")
    write(file, content)
    close(file)
end
make_dzn(100, 200)

In [ ]:
function make_dzn_trip(n::Int, m::Int)
    profits = rand(DiscreteUniform(10, 1000), n)
    weights = rand(DiscreteUniform(10, 100), n)
    capacities = rand(DiscreteUniform(150, 300), m)

    content = """
    n = $n;
    m = $m;
    capacities = $capacities;
    profits = $profits;
    weights = $weights;
    """

    file = open("knapsack_triple_generated.dzn", "w+")
    write(file, content)
    close(file)
end

make_dzn_trip (generic function with 1 method)

In [7]:
make_dzn_trip(100,3)